# 24 — Assembly101 Qwen3.5 Pseudo-Text Extraction

This notebook implements the supervisor's proposed inference-time text path:

```text
raw Assembly101 video
-> 16 temporally ordered frames
-> Qwen3.5-2B
-> 16 short captions + 16 closed-set Assembly101 actions
-> existing CLIP action-text prototypes
-> pseudo-text tensor [512, 16]
```

The resulting tensors have the same shape and normalization as the privileged
text input used by the notebook-18 teacher. Notebook 25 will evaluate that
teacher with pseudo-text and will use **validation F1@50** for model/prompt
selection.

The safe default is a five-sequence validation smoke test. No ground-truth
labels are read, no test examples are processed, and every result is saved
immediately so interrupted Colab sessions can resume.


## Experimental contract

| Item | Fixed choice |
|---|---|
| VLM | `Qwen/Qwen3.5-2B` |
| Input | 16 uniformly spaced frames from each annotated sequence interval |
| Prompt | chronological JSON; caption plus one of the 202 official coarse actions |
| Generation | greedy/non-thinking, up to two format-validation attempts |
| Text representation | existing normalized CLIP ViT-B/16 action prototypes |
| Output | one `[512, 16]` NumPy array per sequence |
| Default split | validation only |
| Ground truth | deliberately not loaded in this notebook |
| Test policy | locked; enable only after Notebook 25 fixes the validation protocol |

A closed label vocabulary is used because the existing teacher was trained on
those exact CLIP action-text prototypes. The free-form captions are retained
for qualitative analysis, but they are not yet used as the teacher input.


## 1. Mount Google Drive


In [15]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


## 2. Install dependencies

Qwen3.5 requires a recent Transformers release. Restart the runtime if Colab
reports that an already-imported package must be reloaded, then continue from
the imports cell.


In [16]:
# %pip install -q -U \
#   "transformers>=5.14.0,<6" \
#   accelerate safetensors \
#   "huggingface_hub[hf_xet]" \
#   decord rapidfuzz pillow "pandas==2.2.2" tqdm

## 3. Imports and hardware audit


In [17]:
from pathlib import Path
from datetime import datetime, timezone

import gc
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import tempfile
import time
import traceback

import numpy as np
import pandas as pd

import torch
from PIL import Image, ImageOps
from decord import VideoReader, cpu
from huggingface_hub import (
    get_token,
    hf_hub_download,
    login,
    notebook_login,
)
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm
from transformers import AutoModelForMultimodalLM, AutoProcessor

os.environ["TOKENIZERS_PARALLELISM"] = "false"
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 240)

if not torch.cuda.is_available():
    raise RuntimeError(
        "Notebook 24 requires a CUDA GPU. In Colab select "
        "Runtime -> Change runtime type -> GPU."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_TOTAL_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
SUPPORTS_BF16 = bool(torch.cuda.is_bf16_supported())
MODEL_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("GPU:", GPU_NAME)
print("GPU memory:", f"{GPU_TOTAL_GIB:.2f} GiB")
print("Selected dtype:", MODEL_DTYPE)
subprocess.run(["nvidia-smi"], check=False)

if GPU_TOTAL_GIB < 10:
    raise RuntimeError(
        "The default Qwen3.5-2B pilot expects roughly 10 GiB or more "
        "of GPU memory. Use a larger GPU or explicitly switch the model "
        "to Qwen/Qwen3.5-0.8B and record that protocol change."
    )


Python: 3.12.13
PyTorch: 2.11.0+cu128
GPU: Tesla T4
GPU memory: 14.56 GiB
Selected dtype: torch.bfloat16


## 4. Configuration


In [18]:
# ---------------------------------------------------------------
# Safe experiment controls
# ---------------------------------------------------------------
# RUN_MODE = "smoke"
RUN_MODE = "validation"

TARGET_SPLIT = "validation"
ALLOW_TEST_GENERATION = False

RUN_CONFIGS = {
    "smoke": {
        "cohort_size": 5,
        "max_recordings_per_session": 5,
        "delete_new_raw_after_success": True,
    },
    "validation": {
        "cohort_size": None,
        "max_recordings_per_session": 10,
        "delete_new_raw_after_success": True,
    },
}

if RUN_MODE not in RUN_CONFIGS:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

CFG = RUN_CONFIGS[RUN_MODE]
FORCE_REGENERATE = False
DELETE_TEMP_FRAMES = True
MAX_SESSION_HOURS = 7.0
STOP_BUFFER_MINUTES = 40

# ---------------------------------------------------------------
# VLM and temporal protocol
# ---------------------------------------------------------------
QWEN_MODEL_ID = "Qwen/Qwen3.5-2B"
EXPERIMENT_ID = "qwen35_2b_closedset_temporal_v1"
PROMPT_VERSION = "closedset_temporal_v1"

NUM_TEMPORAL_STEPS = 16
FRAME_SIDE = 224
PROCESSOR_MAX_IMAGE_PIXELS = FRAME_SIDE * FRAME_SIDE
JPEG_QUALITY = 92

MAX_NEW_TOKENS = 768
MAX_GENERATION_ATTEMPTS = 2
MIN_FUZZY_LABEL_SCORE = 90.0
MIN_ID_LABEL_CONSISTENCY = 70.0

# ---------------------------------------------------------------
# Assembly101 download protocol
# ---------------------------------------------------------------
REPO_ID = "cvml-nus/assembly101"
REPO_TYPE = "dataset"
MAX_DOWNLOAD_ATTEMPTS = 3
RETRY_SLEEP_SECONDS = 10

# ---------------------------------------------------------------
# Project paths: identical roots to notebooks 13--23
# ---------------------------------------------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")
ASSEMBLY_ROOT = DRIVE_ROOT / "assembly101"
MSTCN_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "assembly101"
    / "coarse_mstcn_format"
)
FEATURE_ROOT = MSTCN_ROOT / "streaming_visual_features_v1"
CLIP_ROOT = FEATURE_ROOT / "clip_vitb16"

SEQUENCE_MANIFEST_PATH = (
    MSTCN_ROOT / "procedurevrl_extraction_manifest_v1.csv"
)
RECORDING_MANIFEST_PATH = (
    MSTCN_ROOT / "recording_download_manifest_v1.csv"
)
REMOTE_SIZE_INVENTORY_PATH = (
    MSTCN_ROOT / "v1_download" / "remote_size_inventory_v1.csv"
)
NOTEBOOK14_SUMMARY_PATH = (
    MSTCN_ROOT
    / "v1_download"
    / "assembly101_v1_download_summary.json"
)
TEXT_EMBEDDING_PATH = (
    CLIP_ROOT
    / "text_embeddings"
    / "assembly101_coarse_clip_vitb16_text_embeddings.npy"
)
TEXT_METADATA_PATH = (
    CLIP_ROOT
    / "text_embeddings"
    / "assembly101_coarse_clip_vitb16_text_metadata.csv"
)

OUT_ROOT = (
    FEATURE_ROOT
    / "runs"
    / "24_qwen35_pseudotext"
    / EXPERIMENT_ID
)
RAW_RESPONSE_DIR = OUT_ROOT / "raw_responses"
PARSED_DIR = OUT_ROOT / "parsed_predictions"
EMBEDDING_DIR = OUT_ROOT / "pseudotext_embeddings"
CAPTION_DIR = OUT_ROOT / "captions"
MANIFEST_DIR = OUT_ROOT / "manifests"
LOG_DIR = OUT_ROOT / "logs"
TEMP_FRAME_ROOT = Path("/content/qwen35_pseudotext_frames")

for path in [
    OUT_ROOT,
    RAW_RESPONSE_DIR,
    PARSED_DIR,
    EMBEDDING_DIR,
    CAPTION_DIR,
    MANIFEST_DIR,
    LOG_DIR,
    TEMP_FRAME_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

if TARGET_SPLIT.lower() == "test" and not ALLOW_TEST_GENERATION:
    raise RuntimeError(
        "Test generation is locked. First freeze the prompt and selection "
        "protocol on validation in Notebook 25, then explicitly set "
        "ALLOW_TEST_GENERATION=True."
    )

print("RUN_MODE:", RUN_MODE)
print("TARGET_SPLIT:", TARGET_SPLIT)
print("QWEN_MODEL_ID:", QWEN_MODEL_ID)
print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("OUT_ROOT:", OUT_ROOT)


RUN_MODE: validation
TARGET_SPLIT: validation
QWEN_MODEL_ID: Qwen/Qwen3.5-2B
EXPERIMENT_ID: qwen35_2b_closedset_temporal_v1
OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1


## 5. Authenticate with Hugging Face


In [19]:
def authenticate_huggingface():
    token = None

    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    if token:
        login(token=token, add_to_git_credential=False)
        print("Authenticated using the Colab secret HF_TOKEN.")
        return

    if get_token() is not None:
        print("A Hugging Face token is already available.")
        return

    notebook_login(skip_if_logged_in=True)


authenticate_huggingface()

if get_token() is None:
    raise RuntimeError("Hugging Face authentication did not complete.")


Authenticated using the Colab secret HF_TOKEN.


## 6. Load and validate the existing Assembly101 artifacts


In [20]:
required_paths = {
    "sequence manifest": SEQUENCE_MANIFEST_PATH,
    "recording manifest": RECORDING_MANIFEST_PATH,
    "remote-size inventory": REMOTE_SIZE_INVENTORY_PATH,
    "notebook-14 summary": NOTEBOOK14_SUMMARY_PATH,
    "CLIP action-text embeddings": TEXT_EMBEDDING_PATH,
    "CLIP action-text metadata": TEXT_METADATA_PATH,
}

for name, path in required_paths.items():
    print(f"{name}: {path} -> exists={path.exists()}")
    if not path.exists():
        raise FileNotFoundError(f"Missing prerequisite: {name}: {path}")

sequences = pd.read_csv(SEQUENCE_MANIFEST_PATH)
recordings = pd.read_csv(RECORDING_MANIFEST_PATH)
remote_sizes = pd.read_csv(REMOTE_SIZE_INVENTORY_PATH)
notebook14_summary = json.loads(
    NOTEBOOK14_SUMMARY_PATH.read_text(encoding="utf-8")
)

required_sequence_columns = {
    "sequence_id",
    "split",
    "recording_name",
    "video_remote_path",
    "video_local_path",
    "clip_start_seconds",
    "clip_end_seconds",
}
required_recording_columns = {
    "recording_name",
    "video_remote_path",
    "video_local_path",
}

missing_sequence_columns = required_sequence_columns - set(sequences.columns)
missing_recording_columns = required_recording_columns - set(recordings.columns)

if missing_sequence_columns:
    raise KeyError(
        f"Sequence manifest is missing: {sorted(missing_sequence_columns)}"
    )
if missing_recording_columns:
    raise KeyError(
        f"Recording manifest is missing: {sorted(missing_recording_columns)}"
    )
if "remote_size_bytes" not in remote_sizes.columns:
    raise KeyError("Remote-size inventory has no remote_size_bytes column.")

size_key = (
    "recording_name"
    if "recording_name" in remote_sizes.columns
    else "video_remote_path"
)
recordings = recordings.drop(
    columns=["remote_size_bytes", "remote_size_gib"],
    errors="ignore",
).merge(
    remote_sizes[
        [size_key, "remote_size_bytes"]
    ].drop_duplicates(size_key),
    on=size_key,
    how="left",
    validate="one_to_one",
)

if recordings["remote_size_bytes"].isna().any():
    raise RuntimeError("Some required recordings have no remote size.")

recordings["remote_size_bytes"] = recordings[
    "remote_size_bytes"
].astype("int64")
recordings["remote_size_gib"] = (
    recordings["remote_size_bytes"] / 1024**3
)

if sequences["sequence_id"].duplicated().any():
    raise ValueError("Duplicate sequence IDs in the extraction manifest.")
if recordings["recording_name"].duplicated().any():
    raise ValueError("Duplicate recording names in the recording manifest.")

def canonical_split(value):
    value = str(value).strip().lower()
    if value in {"val", "valid", "validation"}:
        return "validation"
    if value in {"test", "testing"}:
        return "test"
    if value in {"train", "training"}:
        return "train"
    return value


sequences["canonical_split"] = sequences["split"].map(canonical_split)
TARGET_SPLIT_CANONICAL = canonical_split(TARGET_SPLIT)
PINNED_REVISION = str(notebook14_summary["pinned_revision"])

print("Pinned Assembly101 revision:", PINNED_REVISION)
print("Sequences:", len(sequences))
print("Recordings:", len(recordings))
print("Split counts:")
display(sequences["canonical_split"].value_counts().rename("count"))


sequence manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/procedurevrl_extraction_manifest_v1.csv -> exists=True
recording manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/recording_download_manifest_v1.csv -> exists=True
remote-size inventory: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/remote_size_inventory_v1.csv -> exists=True
notebook-14 summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/v1_download/assembly101_v1_download_summary.json -> exists=True
CLIP action-text embeddings: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16/text_embeddings/assembly101_coarse_clip_vitb16_text_embeddings.npy -> exists=True
CLIP action-text metadata: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assem

,count
canonical_split,
train,393
test,167
validation,120


## 7. Load the 202 closed-set text prototypes


In [21]:
text_embeddings = np.load(TEXT_EMBEDDING_PATH).astype(np.float32)
text_metadata = pd.read_csv(TEXT_METADATA_PATH)

required_text_columns = {"model_class_id", "action_cls"}
missing_text_columns = required_text_columns - set(text_metadata.columns)
if missing_text_columns:
    raise KeyError(
        f"Text metadata is missing: {sorted(missing_text_columns)}"
    )

text_metadata = (
    text_metadata
    .sort_values("model_class_id")
    .reset_index(drop=True)
)

EXPECTED_NUM_CLASSES = 202
EXPECTED_TEXT_DIM = 512

if text_embeddings.shape != (EXPECTED_NUM_CLASSES, EXPECTED_TEXT_DIM):
    raise ValueError(
        f"Unexpected text embedding shape: {text_embeddings.shape}"
    )
if not np.array_equal(
    text_metadata["model_class_id"].to_numpy(),
    np.arange(EXPECTED_NUM_CLASSES),
):
    raise ValueError("Text metadata must cover class IDs 0..201 in order.")

prototype_norms = np.linalg.norm(text_embeddings, axis=1, keepdims=True)
text_embeddings = (
    text_embeddings / np.maximum(prototype_norms, 1e-12)
).astype(np.float32)

ACTION_LABELS = text_metadata["action_cls"].astype(str).tolist()
ID_TO_LABEL = dict(enumerate(ACTION_LABELS))

def normalize_label(value):
    value = str(value).strip().casefold()
    for character in ["_", "-", "/", "(", ")", ":", ","]:
        value = value.replace(character, " ")
    return " ".join(value.split())


NORMALIZED_TO_ID = {
    normalize_label(label): class_id
    for class_id, label in ID_TO_LABEL.items()
}
NORMALIZED_LABELS = list(NORMALIZED_TO_ID.keys())

catalog_path = OUT_ROOT / "assembly101_closed_label_catalog.csv"
text_metadata.to_csv(catalog_path, index=False)

print("Text prototypes:", text_embeddings.shape)
print(
    "Norm min/max:",
    float(np.linalg.norm(text_embeddings, axis=1).min()),
    float(np.linalg.norm(text_embeddings, axis=1).max()),
)
print("Saved catalog:", catalog_path)
display(text_metadata.head(20))


Text prototypes: (202, 512)
Norm min/max: 0.9999998807907104 1.0000001192092896
Saved catalog: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/assembly101_closed_label_catalog.csv


,model_class_id,official_action_id,action_cls,verb_cls,noun_cls,prompt,clip_model
0,0,0,inspect toy,inspect,toy,a video of a person performing the action: inspect toy,ViT-B/16
1,1,1,attach cabin,attach,cabin,a video of a person performing the action: attach cabin,ViT-B/16
2,2,2,detach cabin,detach,cabin,a video of a person performing the action: detach cabin,ViT-B/16
3,3,3,detach wheel,detach,wheel,a video of a person performing the action: detach wheel,ViT-B/16
4,4,4,attach wheel,attach,wheel,a video of a person performing the action: attach wheel,ViT-B/16
5,5,5,screw chassis,screw,chassis,a video of a person performing the action: screw chassis,ViT-B/16
6,6,6,demonstrate functionality,demonstrate,functionality,a video of a person performing the action: demonstrate functionality,ViT-B/16
7,7,7,unscrew chassis,unscrew,chassis,a video of a person performing the action: unscrew chassis,ViT-B/16
8,8,8,attach interior,attach,interior,a video of a person performing the action: attach interior,ViT-B/16
9,9,9,detach roof,detach,roof,a video of a person performing the action: detach roof,ViT-B/16


## 8. Freeze the prompt and select a resumable cohort


In [48]:
LABEL_CATALOG_TEXT = "\n".join(
    f"{class_id}: {label}"
    for class_id, label in ID_TO_LABEL.items()
)

BASE_PROMPT = (
    "You are annotating a short egocentric assembly video.\n"
    f"The input contains exactly {NUM_TEMPORAL_STEPS} frames in chronological order,\n"
    "uniformly sampled from one sequence. For every temporal position, describe the\n"
    "main visible action and choose exactly one class from the allowed Assembly101\n"
    "catalog below. Use only visual evidence from the frames.\n\n"
    "Return JSON only, without Markdown or explanations, using this exact schema:\n"
    '{"steps":[\n'
    '  {"step":0,"caption":"short visible action","label_id":0,"label":"exact catalog label"},\n'
    "  ...,\n"
    '  {"step":15,"caption":"short visible action","label_id":0,"label":"exact catalog label"}\n'
    "]}\n\n"
    "Rules:\n"
    "- Return exactly 16 objects with step values 0 through 15, once each.\n"
    "- Each caption must be concise, concrete, and no longer than 12 words.\n"
    "- label_id must be an integer from 0 to 201.\n"
    "- label must match the catalog string for label_id exactly.\n"
    "- If the action is uncertain, choose the closest visible action; do not invent\n"
    "  hidden objects or use future knowledge.\n\n"
    "Allowed catalog:\n"
    + LABEL_CATALOG_TEXT
).strip()

PROMPT_HASH = hashlib.sha256(BASE_PROMPT.encode("utf-8")).hexdigest()
(OUT_ROOT / "prompt.txt").write_text(BASE_PROMPT, encoding="utf-8")

frozen_config = {
    "experiment_id": EXPERIMENT_ID,
    "prompt_version": PROMPT_VERSION,
    "prompt_sha256": PROMPT_HASH,
    "model_id": QWEN_MODEL_ID,
    "generation": {
        "do_sample": False,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_attempts": MAX_GENERATION_ATTEMPTS,
    },
    "frames": {
        "num_temporal_steps": NUM_TEMPORAL_STEPS,
        "frame_side": FRAME_SIDE,
        "sampling": "uniform_bin_centers",
    },
    "target_split": TARGET_SPLIT_CANONICAL,
    "assembly101_revision": PINNED_REVISION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(OUT_ROOT / "frozen_config.json").write_text(
    json.dumps(frozen_config, indent=2),
    encoding="utf-8",
)

def saved_output_is_valid(sequence_id):
    metadata_path = PARSED_DIR / f"{sequence_id}.json"
    embedding_path = EMBEDDING_DIR / f"{sequence_id}.npy"
    caption_path = CAPTION_DIR / f"{sequence_id}.txt"

    if not (metadata_path.exists() and embedding_path.exists() and caption_path.exists()):
        return False

    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        embedding = np.load(embedding_path)
        return bool(
            metadata.get("prompt_sha256") == PROMPT_HASH
            and metadata.get("model_id") == QWEN_MODEL_ID
            and len(metadata.get("steps", [])) == NUM_TEMPORAL_STEPS
            and embedding.shape == (EXPECTED_TEXT_DIM, NUM_TEMPORAL_STEPS)
            and np.isfinite(embedding).all()
        )
    except Exception:
        return False


target_sequences = sequences[
    sequences["canonical_split"] == TARGET_SPLIT_CANONICAL
].copy()

if target_sequences.empty:
    raise RuntimeError(f"No sequences found for split {TARGET_SPLIT_CANONICAL!r}.")

target_sequences = target_sequences.merge(
    recordings[
        ["recording_name", "remote_size_bytes", "remote_size_gib"]
    ],
    on="recording_name",
    how="left",
    validate="many_to_one",
).sort_values(
    [
        "remote_size_bytes",
        "recording_name",
        "clip_start_seconds",
        "sequence_id",
    ]
).reset_index(drop=True)

# Smoke mode is a fixed cohort, not the next five unfinished examples.
# Therefore rerunning the completed smoke notebook performs no new work.
cohort_size = CFG["cohort_size"]
cohort = (
    target_sequences.head(cohort_size).copy()
    if cohort_size is not None
    else target_sequences.copy()
)
cohort["complete_before_run"] = cohort["sequence_id"].map(
    saved_output_is_valid
)

if FORCE_REGENERATE:
    selected_sequences = cohort.copy()
else:
    selected_sequences = cohort[~cohort["complete_before_run"]].copy()

recording_queue = (
    selected_sequences
    .groupby("recording_name", as_index=False)
    .agg(
        pending_sequences=("sequence_id", "size"),
        remote_size_bytes=("remote_size_bytes", "first"),
        remote_size_gib=("remote_size_gib", "first"),
    )
    .sort_values(["remote_size_bytes", "recording_name"])
    .head(CFG["max_recordings_per_session"])
    .reset_index(drop=True)
)

print("Prompt SHA256:", PROMPT_HASH)
print("Target split sequences:", len(target_sequences))
print("Fixed cohort:", len(cohort))
print("Already complete:", int(cohort["complete_before_run"].sum()))
print("Pending in cohort:", len(selected_sequences))
print("Recordings scheduled this session:", len(recording_queue))
display(recording_queue)
display(
    cohort[
        [
            "sequence_id",
            "recording_name",
            "clip_start_seconds",
            "clip_end_seconds",
            "complete_before_run",
        ]
    ].head(30)
)


Prompt SHA256: 08047fb92444d9b043a3894ab2c9f91e516a9e466d02e52f67446ed9551701b6
Target split sequences: 120
Fixed cohort: 120
Already complete: 119
Pending in cohort: 1
Recordings scheduled this session: 1


,recording_name,pending_sequences,remote_size_bytes,remote_size_gib
0,nusar-2021_action_both_9013-c09c_9013_user_id_2021-02-24_113951,1,823561938,0.767002


,sequence_id,recording_name,clip_start_seconds,clip_end_seconds,complete_before_run
0,disassembly_nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,4.833333,90.666667,True
1,assembly_nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,93.933333,265.133333,True
2,disassembly_nusar-2021_action_both_9046-b06b_9046_user_id_2021-02-22_105953,nusar-2021_action_both_9046-b06b_9046_user_id_2021-02-22_105953,4.333333,121.700000,True
3,assembly_nusar-2021_action_both_9046-b06b_9046_user_id_2021-02-22_105953,nusar-2021_action_both_9046-b06b_9046_user_id_2021-02-22_105953,126.033333,248.033333,True
4,disassembly_nusar-2021_action_both_9022-a18_9022_user_id_2021-02-23_104757,nusar-2021_action_both_9022-a18_9022_user_id_2021-02-23_104757,4.366667,105.566667,True
5,assembly_nusar-2021_action_both_9022-a18_9022_user_id_2021-02-23_104757,nusar-2021_action_both_9022-a18_9022_user_id_2021-02-23_104757,108.066667,242.866667,True
6,disassembly_nusar-2021_action_both_9021-c10a_9021_user_id_2021-02-23_100458,nusar-2021_action_both_9021-c10a_9021_user_id_2021-02-23_100458,6.233333,110.366667,True
7,assembly_nusar-2021_action_both_9021-c10a_9021_user_id_2021-02-23_100458,nusar-2021_action_both_9021-c10a_9021_user_id_2021-02-23_100458,116.033333,260.700000,True
8,disassembly_nusar-2021_action_both_9023-c09c_9023_user_id_2021-02-23_134459,nusar-2021_action_both_9023-c09c_9023_user_id_2021-02-23_134459,6.533333,142.933333,True
9,assembly_nusar-2021_action_both_9023-c09c_9023_user_id_2021-02-23_134459,nusar-2021_action_both_9023-c09c_9023_user_id_2021-02-23_134459,148.300000,365.666667,True


## 9. Load Qwen3.5-2B

The processor is capped at `224 x 224` visual pixels per sampled frame.
This is deliberately conservative for a 16 GB T4. If this cell fails with
OOM, do not silently change the model within the same experiment directory;
create a new `EXPERIMENT_ID` for a smaller model or lower resolution.


In [23]:
torch.cuda.empty_cache()
gc.collect()

if recording_queue.empty:
    processor = None
    model = None
    MODEL_DEVICE = torch.device("cuda")
    print("No pending recordings: model loading skipped.")
else:
    processor = AutoProcessor.from_pretrained(
        QWEN_MODEL_ID,
        max_image_size={"longest_edge": PROCESSOR_MAX_IMAGE_PIXELS},
    )
    processor.tokenizer.padding_side = "left"

    model = AutoModelForMultimodalLM.from_pretrained(
        QWEN_MODEL_ID,
        dtype=MODEL_DTYPE,
        device_map="auto",
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )
    model.eval()

    MODEL_DEVICE = next(model.parameters()).device
    parameter_count = sum(parameter.numel() for parameter in model.parameters())

    print("Model device:", MODEL_DEVICE)
    print("Parameters:", f"{parameter_count / 1e9:.3f} B")
    print(
        "Allocated after load:",
        f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB",
    )


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Model device: cuda:0
Parameters: 2.213 B
Allocated after load: 4.13 GiB


## 10. Frame sampling and temporary media preparation


In [24]:
CENTER_POSITIONS = (
    np.arange(NUM_TEMPORAL_STEPS, dtype=np.float64) + 0.5
) / NUM_TEMPORAL_STEPS

def letterbox_frame(frame, side=FRAME_SIDE):
    image = Image.fromarray(frame).convert("RGB")
    image = ImageOps.contain(image, (side, side), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (side, side), color=(0, 0, 0))
    offset = ((side - image.width) // 2, (side - image.height) // 2)
    canvas.paste(image, offset)
    return canvas


def sample_sequence_frames(
    video_path,
    clip_start_seconds,
    clip_end_seconds,
):
    video_reader = VideoReader(str(video_path), ctx=cpu(0))
    total_frames = len(video_reader)
    fps = float(video_reader.get_avg_fps())

    if total_frames <= 0 or fps <= 0:
        raise RuntimeError(
            f"Invalid video metadata: frames={total_frames}, fps={fps}"
        )

    clip_start_seconds = float(clip_start_seconds)
    clip_end_seconds = float(clip_end_seconds)
    if not (0 <= clip_start_seconds < clip_end_seconds):
        raise ValueError(
            f"Invalid clip interval: {clip_start_seconds}..{clip_end_seconds}"
        )

    absolute_times = (
        clip_start_seconds
        + CENTER_POSITIONS * (clip_end_seconds - clip_start_seconds)
    )
    frame_indices = np.rint(absolute_times * fps).astype(np.int64)
    frame_indices = np.clip(frame_indices, 0, total_frames - 1)
    frames = video_reader.get_batch(frame_indices.tolist()).asnumpy()

    return {
        "frames": frames,
        "frame_indices": frame_indices,
        "absolute_times": absolute_times,
        "fps": fps,
        "total_frames": total_frames,
    }


def save_temporary_frames(sequence_id, frames):
    sequence_dir = TEMP_FRAME_ROOT / str(sequence_id)
    if sequence_dir.exists():
        shutil.rmtree(sequence_dir)
    sequence_dir.mkdir(parents=True, exist_ok=True)

    frame_paths = []
    for step_index, frame in enumerate(frames):
        frame_path = sequence_dir / f"frame_{step_index:02d}.jpg"
        letterbox_frame(frame).save(
            frame_path,
            format="JPEG",
            quality=JPEG_QUALITY,
            optimize=True,
        )
        frame_paths.append(frame_path)

    return sequence_dir, frame_paths


print("Temporal centers:", CENTER_POSITIONS.tolist())


Temporal centers: [0.03125, 0.09375, 0.15625, 0.21875, 0.28125, 0.34375, 0.40625, 0.46875, 0.53125, 0.59375, 0.65625, 0.71875, 0.78125, 0.84375, 0.90625, 0.96875]


## 11. Strict JSON and label validation


In [25]:
def extract_first_json_object(text):
    decoder = json.JSONDecoder()
    for start_index, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[start_index:])
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            continue
    raise ValueError("No valid JSON object found in the model response.")


def resolve_action_label(provided_id, provided_label):
    normalized_label = normalize_label(provided_label)
    exact_id = NORMALIZED_TO_ID.get(normalized_label)

    try:
        integer_id = int(provided_id)
        valid_id = 0 <= integer_id < EXPECTED_NUM_CLASSES
    except Exception:
        integer_id = None
        valid_id = False

    if exact_id is not None:
        return {
            "model_class_id": int(exact_id),
            "action_cls": ID_TO_LABEL[int(exact_id)],
            "match_method": (
                "exact_label_and_id"
                if valid_id and integer_id == exact_id
                else "exact_label_repaired_id"
            ),
            "match_score": 100.0,
            "provided_label_id": provided_id,
            "provided_label": str(provided_label),
        }

    if valid_id:
        expected_normalized = normalize_label(ID_TO_LABEL[integer_id])
        consistency = float(fuzz.WRatio(normalized_label, expected_normalized))
        if consistency >= MIN_ID_LABEL_CONSISTENCY:
            return {
                "model_class_id": int(integer_id),
                "action_cls": ID_TO_LABEL[int(integer_id)],
                "match_method": "valid_id_repaired_label",
                "match_score": consistency,
                "provided_label_id": provided_id,
                "provided_label": str(provided_label),
            }

    fuzzy_match = process.extractOne(
        normalized_label,
        NORMALIZED_LABELS,
        scorer=fuzz.WRatio,
    )
    if fuzzy_match is not None:
        matched_normalized, score, _ = fuzzy_match
        if float(score) >= MIN_FUZZY_LABEL_SCORE:
            matched_id = NORMALIZED_TO_ID[matched_normalized]
            return {
                "model_class_id": int(matched_id),
                "action_cls": ID_TO_LABEL[int(matched_id)],
                "match_method": "fuzzy_label_repair",
                "match_score": float(score),
                "provided_label_id": provided_id,
                "provided_label": str(provided_label),
            }

    raise ValueError(
        f"Could not map label_id={provided_id!r}, label={provided_label!r}."
    )


def parse_and_validate_response(raw_text):
    payload = extract_first_json_object(raw_text)
    raw_steps = payload.get("steps")

    if not isinstance(raw_steps, list):
        raise ValueError("JSON field 'steps' is not a list.")
    if len(raw_steps) != NUM_TEMPORAL_STEPS:
        raise ValueError(
            f"Expected {NUM_TEMPORAL_STEPS} steps, received {len(raw_steps)}."
        )

    parsed_steps = []
    for raw_step in raw_steps:
        if not isinstance(raw_step, dict):
            raise ValueError("Every step must be a JSON object.")

        try:
            step_index = int(raw_step.get("step"))
        except Exception as exc:
            raise ValueError(f"Invalid step index: {raw_step.get('step')!r}") from exc

        caption = " ".join(str(raw_step.get("caption", "")).split())
        if not caption:
            raise ValueError(f"Step {step_index} has an empty caption.")

        resolved = resolve_action_label(
            raw_step.get("label_id"),
            raw_step.get("label", ""),
        )
        parsed_steps.append(
            {
                "step": step_index,
                "caption": caption,
                **resolved,
            }
        )

    parsed_steps = sorted(parsed_steps, key=lambda row: row["step"])
    observed_indices = [row["step"] for row in parsed_steps]
    expected_indices = list(range(NUM_TEMPORAL_STEPS))
    if observed_indices != expected_indices:
        raise ValueError(
            f"Step indices must be exactly {expected_indices}; got {observed_indices}."
        )

    return parsed_steps


print("Response parser ready.")


Response parser ready.


## 12. Qwen inference with a video-frame-list fallback


In [26]:
def build_messages(frame_paths, prompt, media_mode):
    if media_mode == "video_frame_list":
        visual_content = [
            {
                "type": "video",
                "path": [str(path) for path in frame_paths],
            }
        ]
    elif media_mode == "ordered_images":
        visual_content = [
            {"type": "image", "path": str(path)}
            for path in frame_paths
        ]
    else:
        raise ValueError(f"Unknown media mode: {media_mode}")

    return [
        {
            "role": "user",
            "content": visual_content + [{"type": "text", "text": prompt}],
        }
    ]


def prepare_multimodal_inputs(frame_paths, prompt):
    errors = []
    for media_mode in ["video_frame_list", "ordered_images"]:
        messages = build_messages(frame_paths, prompt, media_mode)
        try:
            kwargs = {
                "add_generation_prompt": True,
                "tokenize": True,
                "return_dict": True,
                "return_tensors": "pt",
            }
            if media_mode == "video_frame_list":
                kwargs["video_load_backend"] = "decord"

            inputs = processor.apply_chat_template(messages, **kwargs)
            return inputs.to(MODEL_DEVICE), media_mode
        except Exception as exc:
            errors.append(
                f"{media_mode}: {type(exc).__name__}: {exc}"
            )

    raise RuntimeError(
        "Both multimodal input modes failed:\n" + "\n".join(errors)
    )


def generate_once(frame_paths, prompt):
    inputs, media_mode = prepare_multimodal_inputs(frame_paths, prompt)
    input_tokens = int(inputs["input_ids"].shape[-1])

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()
    generation_seconds = time.perf_counter() - started

    continuation = generated_ids[:, input_tokens:]
    raw_text = processor.batch_decode(
        continuation,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return {
        "raw_text": raw_text,
        "media_mode": media_mode,
        "input_tokens": input_tokens,
        "output_tokens": int(continuation.shape[-1]),
        "generation_seconds": float(generation_seconds),
        "peak_gpu_gib": float(
            torch.cuda.max_memory_allocated() / 1024**3
        ),
    }


def generate_validated_steps(sequence_id, frame_paths):
    validation_error = ""

    for attempt in range(1, MAX_GENERATION_ATTEMPTS + 1):
        prompt = BASE_PROMPT
        if attempt > 1:
            prompt += (
                "\n\nYour previous response failed automatic validation because: "
                + validation_error[:600]
                + "\nReturn a corrected JSON object only."
            )

        generation = generate_once(frame_paths, prompt)
        raw_path = (
            RAW_RESPONSE_DIR
            / f"{sequence_id}.attempt_{attempt}.txt"
        )
        raw_path.write_text(generation["raw_text"], encoding="utf-8")

        try:
            steps = parse_and_validate_response(generation["raw_text"])
            return {
                "steps": steps,
                "attempt": attempt,
                "raw_response_path": str(raw_path),
                **generation,
            }
        except Exception as exc:
            validation_error = f"{type(exc).__name__}: {exc}"
            print(
                f"{sequence_id}: attempt {attempt} failed validation: "
                f"{validation_error}"
            )

    raise RuntimeError(
        f"No valid response after {MAX_GENERATION_ATTEMPTS} attempts: "
        f"{validation_error}"
    )


print("Qwen generation functions ready.")


Qwen generation functions ready.


## 13. Download and process the scheduled recordings


In [49]:
def atomic_write_text(path, text):
    path = Path(path)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def atomic_save_npy(path, array):
    path = Path(path)
    temporary_path = path.with_name(path.stem + ".tmp.npy")
    np.save(temporary_path, array)
    os.replace(temporary_path, path)


def download_one_recording(recording_row):
    target_path = Path(recording_row.video_local_path)
    expected_size = int(recording_row.remote_size_bytes)
    existed_before = bool(
        target_path.exists() and target_path.stat().st_size == expected_size
    )

    if existed_before:
        return target_path, False

    if target_path.exists():
        print("Removing incomplete cached video:", target_path)
        target_path.unlink()

    last_error = None
    for attempt in range(1, MAX_DOWNLOAD_ATTEMPTS + 1):
        try:
            print(
                f"Downloading {recording_row.recording_name}, "
                f"attempt {attempt}/{MAX_DOWNLOAD_ATTEMPTS}, "
                f"{expected_size / 1024**3:.3f} GiB"
            )
            returned_path = Path(
                hf_hub_download(
                    repo_id=REPO_ID,
                    repo_type=REPO_TYPE,
                    revision=PINNED_REVISION,
                    filename=recording_row.video_remote_path,
                    local_dir=ASSEMBLY_ROOT,
                    token=True,
                )
            )

            if not target_path.exists():
                raise FileNotFoundError(
                    f"Expected {target_path}, downloader returned {returned_path}."
                )
            if target_path.stat().st_size != expected_size:
                raise IOError(
                    f"Downloaded size {target_path.stat().st_size} "
                    f"!= expected {expected_size}."
                )
            return target_path, True

        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            print("Download attempt failed:", last_error)
            if attempt < MAX_DOWNLOAD_ATTEMPTS:
                time.sleep(RETRY_SLEEP_SECONDS)

    raise RuntimeError(
        f"Could not download {recording_row.recording_name}: {last_error}"
    )


def save_sequence_result(row, sampled, generation):
    class_ids = np.asarray(
        [step["model_class_id"] for step in generation["steps"]],
        dtype=np.int64,
    )
    pseudotext = text_embeddings[class_ids].T.astype(np.float32)

    if pseudotext.shape != (EXPECTED_TEXT_DIM, NUM_TEMPORAL_STEPS):
        raise RuntimeError(f"Unexpected pseudo-text shape: {pseudotext.shape}")
    if not np.isfinite(pseudotext).all():
        raise RuntimeError("Pseudo-text contains NaN or Inf.")

    embedding_path = EMBEDDING_DIR / f"{row.sequence_id}.npy"
    caption_path = CAPTION_DIR / f"{row.sequence_id}.txt"
    metadata_path = PARSED_DIR / f"{row.sequence_id}.json"

    metadata = {
        "sequence_id": str(row.sequence_id),
        "split": TARGET_SPLIT_CANONICAL,
        "recording_name": str(row.recording_name),
        "model_id": QWEN_MODEL_ID,
        "experiment_id": EXPERIMENT_ID,
        "prompt_version": PROMPT_VERSION,
        "prompt_sha256": PROMPT_HASH,
        "assembly101_revision": PINNED_REVISION,
        "num_temporal_steps": NUM_TEMPORAL_STEPS,
        "frame_side": FRAME_SIDE,
        "clip_start_seconds": float(row.clip_start_seconds),
        "clip_end_seconds": float(row.clip_end_seconds),
        "frame_indices": sampled["frame_indices"].astype(int).tolist(),
        "absolute_times_seconds": sampled["absolute_times"].astype(float).tolist(),
        "video_fps": float(sampled["fps"]),
        "video_total_frames": int(sampled["total_frames"]),
        "attempt": int(generation["attempt"]),
        "media_mode": generation["media_mode"],
        "input_tokens": int(generation["input_tokens"]),
        "output_tokens": int(generation["output_tokens"]),
        "generation_seconds": float(generation["generation_seconds"]),
        "peak_gpu_gib": float(generation["peak_gpu_gib"]),
        "raw_response_path": generation["raw_response_path"],
        "embedding_path": str(embedding_path),
        "captions_path": str(caption_path),
        "embedding_shape": list(pseudotext.shape),
        "steps": generation["steps"],
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }

    caption_lines = [
        f"{step['step']:02d}\t{step['caption']}\t{step['action_cls']}"
        for step in generation["steps"]
    ]

    atomic_save_npy(embedding_path, pseudotext)
    atomic_write_text(caption_path, "\n".join(caption_lines) + "\n")
    atomic_write_text(
        metadata_path,
        json.dumps(metadata, indent=2, ensure_ascii=False),
    )
    return metadata


def append_failure(row, exc):
    failure = {
        "sequence_id": str(row.sequence_id),
        "recording_name": str(row.recording_name),
        "error_type": type(exc).__name__,
        "error": str(exc),
        "traceback": traceback.format_exc(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    with (LOG_DIR / "failures.jsonl").open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(failure, ensure_ascii=False) + "\n")


session_started = time.monotonic()
session_deadline = (
    session_started
    + MAX_SESSION_HOURS * 3600
    - STOP_BUFFER_MINUTES * 60
)
session_rows = []

for queue_row in recording_queue.itertuples(index=False):
    if time.monotonic() >= session_deadline:
        print("Stopping before the session safety deadline.")
        break

    recording_row = recordings[
        recordings["recording_name"] == queue_row.recording_name
    ].iloc[0]
    video_path, downloaded_this_run = download_one_recording(recording_row)

    recording_sequences = selected_sequences[
        selected_sequences["recording_name"] == queue_row.recording_name
    ].sort_values(["clip_start_seconds", "sequence_id"])

    for row in tqdm(
        recording_sequences.itertuples(index=False),
        total=len(recording_sequences),
        desc=str(queue_row.recording_name),
    ):
        if not FORCE_REGENERATE and saved_output_is_valid(row.sequence_id):
            continue

        sequence_dir = None
        try:
            sampled = sample_sequence_frames(
                video_path,
                row.clip_start_seconds,
                row.clip_end_seconds,
            )
            sequence_dir, frame_paths = save_temporary_frames(
                row.sequence_id,
                sampled["frames"],
            )
            generation = generate_validated_steps(
                row.sequence_id,
                frame_paths,
            )
            metadata = save_sequence_result(row, sampled, generation)
            session_rows.append(
                {
                    "sequence_id": row.sequence_id,
                    "recording_name": row.recording_name,
                    "status": "completed",
                    "generation_seconds": metadata["generation_seconds"],
                    "attempt": metadata["attempt"],
                    "media_mode": metadata["media_mode"],
                    "error": "",
                }
            )
            print(
                f"{row.sequence_id}: saved [512, 16] in "
                f"{metadata['generation_seconds']:.1f}s"
            )

        except Exception as exc:
            append_failure(row, exc)
            session_rows.append(
                {
                    "sequence_id": row.sequence_id,
                    "recording_name": row.recording_name,
                    "status": "failed",
                    "generation_seconds": np.nan,
                    "attempt": np.nan,
                    "media_mode": "",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            print(f"{row.sequence_id}: FAILED: {type(exc).__name__}: {exc}")

        finally:
            if DELETE_TEMP_FRAMES and sequence_dir is not None and sequence_dir.exists():
                shutil.rmtree(sequence_dir)
            gc.collect()
            torch.cuda.empty_cache()

    planned_recording_ids = cohort.loc[
        cohort["recording_name"] == queue_row.recording_name,
        "sequence_id",
    ].tolist()
    recording_complete = all(
        saved_output_is_valid(sequence_id)
        for sequence_id in planned_recording_ids
    )

    if (
        CFG["delete_new_raw_after_success"]
        and downloaded_this_run
        and recording_complete
        and video_path.exists()
    ):
        video_path.unlink()
        print("Deleted this notebook's completed raw download:", video_path)

session_manifest = pd.DataFrame(session_rows)
session_manifest_path = (
    MANIFEST_DIR
    / f"session_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.csv"
)
session_manifest.to_csv(session_manifest_path, index=False)

print("Session manifest:", session_manifest_path)
display(session_manifest)


nusar-2021_action_both_9013-c09c_9013_user_id_2021-02-24_113951:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


disassembly_nusar-2021_action_both_9013-c09c_9013_user_id_2021-02-24_113951: attempt 1 failed validation: ValueError: Could not map label_id=1, label='move toy truck'.
disassembly_nusar-2021_action_both_9013-c09c_9013_user_id_2021-02-24_113951: attempt 2 failed validation: ValueError: Could not map label_id=0, label='move toy truck'.
disassembly_nusar-2021_action_both_9013-c09c_9013_user_id_2021-02-24_113951: FAILED: RuntimeError: No valid response after 2 attempts: ValueError: Could not map label_id=0, label='move toy truck'.
Session manifest: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/manifests/session_20260818T180650Z.csv


,sequence_id,recording_name,status,generation_seconds,attempt,media_mode,error
0,disassembly_nusar-2021_action_both_9013-c09c_9013_user_id_2021-02-24_113951,nusar-2021_action_both_9013-c09c_9013_user_id_2021-02-24_113951,failed,NaN,NaN,,"RuntimeError: No valid response after 2 attempts: ValueError: Could not map label_id=0, label='move toy truck'."


In [51]:
FAILED_SEQUENCE_ID = (
    "disassembly_nusar-2021_action_both_9013-c09c_"
    "9013_user_id_2021-02-24_113951"
)

raw_paths = sorted(
    RAW_RESPONSE_DIR.glob(
        f"{FAILED_SEQUENCE_ID}.attempt_*.txt"
    )
)

if not raw_paths:
    raise FileNotFoundError(
        f"No raw Qwen responses found for {FAILED_SEQUENCE_ID}"
    )

# Use the latest repair attempt.
raw_response_path = raw_paths[-1]
raw_text = raw_response_path.read_text(encoding="utf-8")
payload = extract_first_json_object(raw_text)
raw_steps = payload.get("steps", [])

if len(raw_steps) != NUM_TEMPORAL_STEPS:
    raise ValueError(
        f"Expected {NUM_TEMPORAL_STEPS} steps, got {len(raw_steps)}"
    )

step_records = {}
step_vectors = {}
fallback_steps = []

for raw_step in raw_steps:
    step_index = int(raw_step["step"])
    caption = " ".join(
        str(raw_step.get("caption", "")).split()
    )

    try:
        resolved = resolve_action_label(
            raw_step.get("label_id"),
            raw_step.get("label", ""),
        )

        vector = text_embeddings[
            resolved["model_class_id"]
        ].astype(np.float32)

        step_record = {
            "step": step_index,
            "caption": caption,
            **resolved,
        }

    except Exception as exc:
        # Conservative fallback: do not invent an Assembly101 class.
        vector = np.zeros(
            EXPECTED_TEXT_DIM,
            dtype=np.float32,
        )

        step_record = {
            "step": step_index,
            "caption": caption,
            "model_class_id": None,
            "action_cls": "__missing_text__",
            "match_method": "zero_step_fallback",
            "match_score": None,
            "provided_label_id": raw_step.get("label_id"),
            "provided_label": str(
                raw_step.get("label", "")
            ),
            "fallback_error": f"{type(exc).__name__}: {exc}",
        }

        fallback_steps.append(step_index)

    step_records[step_index] = step_record
    step_vectors[step_index] = vector

expected_steps = list(range(NUM_TEMPORAL_STEPS))

if sorted(step_records) != expected_steps:
    raise ValueError(
        f"Expected steps {expected_steps}, got {sorted(step_records)}"
    )

steps = [
    step_records[index]
    for index in expected_steps
]

pseudotext = np.stack(
    [
        step_vectors[index]
        for index in expected_steps
    ],
    axis=1,
).astype(np.float32)

assert pseudotext.shape == (
    EXPECTED_TEXT_DIM,
    NUM_TEMPORAL_STEPS,
)
assert np.isfinite(pseudotext).all()

sequence_row = sequences.loc[
    sequences["sequence_id"].astype(str)
    == FAILED_SEQUENCE_ID
].iloc[0]

embedding_path = (
    EMBEDDING_DIR / f"{FAILED_SEQUENCE_ID}.npy"
)
caption_path = (
    CAPTION_DIR / f"{FAILED_SEQUENCE_ID}.txt"
)
metadata_path = (
    PARSED_DIR / f"{FAILED_SEQUENCE_ID}.json"
)

np.save(embedding_path, pseudotext)

caption_path.write_text(
    "\n".join(
        f"{step['step']:02d}\t"
        f"{step['caption']}\t"
        f"{step['action_cls']}"
        for step in steps
    )
    + "\n",
    encoding="utf-8",
)

metadata = {
    "sequence_id": FAILED_SEQUENCE_ID,
    "split": TARGET_SPLIT_CANONICAL,
    "recording_name": str(
        sequence_row["recording_name"]
    ),
    "model_id": QWEN_MODEL_ID,
    "experiment_id": EXPERIMENT_ID,
    "prompt_version": PROMPT_VERSION,
    "prompt_sha256": PROMPT_HASH,
    "assembly101_revision": PINNED_REVISION,
    "num_temporal_steps": NUM_TEMPORAL_STEPS,
    "clip_start_seconds": float(
        sequence_row["clip_start_seconds"]
    ),
    "clip_end_seconds": float(
        sequence_row["clip_end_seconds"]
    ),
    "attempt": 2,
    "media_mode": "ordered_images",
    "generation_seconds": None,
    "raw_response_path": str(raw_response_path),
    "embedding_path": str(embedding_path),
    "captions_path": str(caption_path),
    "embedding_shape": list(pseudotext.shape),
    "fallback_policy": (
        "zero_embedding_for_unmappable_steps_only"
    ),
    "fallback_steps": fallback_steps,
    "ground_truth_used": False,
    "steps": steps,
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

metadata_path.write_text(
    json.dumps(
        metadata,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("Saved repaired sequence:", FAILED_SEQUENCE_ID)
print("Fallback temporal steps:", fallback_steps)
print("Embedding shape:", pseudotext.shape)
print(
    "Output valid:",
    saved_output_is_valid(FAILED_SEQUENCE_ID),
)

Saved repaired sequence: disassembly_nusar-2021_action_both_9013-c09c_9013_user_id_2021-02-24_113951
Fallback temporal steps: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Embedding shape: (512, 16)
Output valid: True


## 14. Rebuild persistent manifests and validate every saved tensor


In [52]:
status_rows = []
step_rows = []

for row in target_sequences.itertuples(index=False):
    sequence_id = str(row.sequence_id)
    valid = saved_output_is_valid(sequence_id)
    status_row = {
        "sequence_id": sequence_id,
        "split": TARGET_SPLIT_CANONICAL,
        "recording_name": str(row.recording_name),
        "in_current_cohort": sequence_id in set(cohort["sequence_id"].astype(str)),
        "complete": valid,
        "embedding_path": str(EMBEDDING_DIR / f"{sequence_id}.npy"),
        "metadata_path": str(PARSED_DIR / f"{sequence_id}.json"),
    }

    if valid:
        metadata = json.loads(
            (PARSED_DIR / f"{sequence_id}.json").read_text(encoding="utf-8")
        )
        embedding = np.load(EMBEDDING_DIR / f"{sequence_id}.npy")
        column_norms = np.linalg.norm(embedding, axis=0)

        status_row.update(
            {
                "generation_seconds": metadata["generation_seconds"],
                "attempt": metadata["attempt"],
                "media_mode": metadata["media_mode"],
                "min_column_norm": float(column_norms.min()),
                "max_column_norm": float(column_norms.max()),
            }
        )

        for step in metadata["steps"]:
            step_rows.append(
                {
                    "sequence_id": sequence_id,
                    "recording_name": str(row.recording_name),
                    **step,
                }
            )

    status_rows.append(status_row)

status_df = pd.DataFrame(status_rows)
steps_df = pd.DataFrame(step_rows)

status_path = MANIFEST_DIR / "sequence_status.csv"
steps_path = MANIFEST_DIR / "step_predictions.csv"
label_histogram_path = MANIFEST_DIR / "predicted_label_histogram.csv"

status_df.to_csv(status_path, index=False)
steps_df.to_csv(steps_path, index=False)

if not steps_df.empty:
    label_histogram = (
        steps_df
        .groupby(["model_class_id", "action_cls"], as_index=False)
        .size()
        .sort_values("size", ascending=False)
        .rename(columns={"size": "predicted_steps"})
    )
else:
    label_histogram = pd.DataFrame(
        columns=["model_class_id", "action_cls", "predicted_steps"]
    )
label_histogram.to_csv(label_histogram_path, index=False)

cohort_ids = set(cohort["sequence_id"].astype(str))
cohort_status = status_df[status_df["sequence_id"].isin(cohort_ids)]
completed_cohort = int(cohort_status["complete"].sum())

if "generation_seconds" in cohort_status.columns:
    generation_times = pd.to_numeric(
        cohort_status.loc[
            cohort_status["complete"],
            "generation_seconds",
        ],
        errors="coerce",
    ).dropna()
else:
    generation_times = pd.Series(dtype=np.float64)

match_counts = (
    steps_df["match_method"].value_counts().to_dict()
    if not steps_df.empty
    else {}
)

summary = {
    "status": (
        "completed"
        if completed_cohort == len(cohort)
        else "partial"
    ),
    "run_mode": RUN_MODE,
    "target_split": TARGET_SPLIT_CANONICAL,
    "experiment_id": EXPERIMENT_ID,
    "model_id": QWEN_MODEL_ID,
    "prompt_version": PROMPT_VERSION,
    "prompt_sha256": PROMPT_HASH,
    "assembly101_revision": PINNED_REVISION,
    "cohort_sequences": int(len(cohort)),
    "completed_cohort_sequences": completed_cohort,
    "remaining_cohort_sequences": int(len(cohort) - completed_cohort),
    "total_saved_target_sequences": int(status_df["complete"].sum()),
    "saved_steps": int(len(steps_df)),
    "match_methods": {str(key): int(value) for key, value in match_counts.items()},
    "generation_seconds_mean": (
        float(generation_times.mean()) if len(generation_times) else None
    ),
    "generation_seconds_median": (
        float(generation_times.median()) if len(generation_times) else None
    ),
    "gpu_name": GPU_NAME,
    "gpu_total_gib": float(GPU_TOTAL_GIB),
    "dtype": str(MODEL_DTYPE),
    "output_shape_per_sequence": [EXPECTED_TEXT_DIM, NUM_TEMPORAL_STEPS],
    "ground_truth_loaded": False,
    "updated_utc": datetime.now(timezone.utc).isoformat(),
    "artifacts": {
        "status_csv": str(status_path),
        "steps_csv": str(steps_path),
        "label_histogram_csv": str(label_histogram_path),
        "embedding_dir": str(EMBEDDING_DIR),
        "parsed_dir": str(PARSED_DIR),
    },
}

summary_path = OUT_ROOT / "final_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("Saved:", summary_path)
display(cohort_status)
display(label_histogram.head(30))

if RUN_MODE == "smoke" and completed_cohort != len(cohort):
    raise RuntimeError(
        "Smoke test is incomplete. Inspect logs/failures.jsonl and rerun "
        "before switching RUN_MODE to 'validation'."
    )


{
  "status": "completed",
  "run_mode": "validation",
  "target_split": "validation",
  "experiment_id": "qwen35_2b_closedset_temporal_v1",
  "model_id": "Qwen/Qwen3.5-2B",
  "prompt_version": "closedset_temporal_v1",
  "prompt_sha256": "08047fb92444d9b043a3894ab2c9f91e516a9e466d02e52f67446ed9551701b6",
  "assembly101_revision": "bfc15ea5e3f0bc8f8c232af6c1b45aa137a9d967",
  "cohort_sequences": 120,
  "completed_cohort_sequences": 120,
  "remaining_cohort_sequences": 0,
  "total_saved_target_sequences": 120,
  "saved_steps": 1920,
  "match_methods": {
    "exact_label_and_id": 1829,
    "exact_label_repaired_id": 75,
    "zero_step_fallback": 16
  },
  "generation_seconds_mean": 71.54650491415973,
  "generation_seconds_median": 70.01750240599995,
  "gpu_name": "Tesla T4",
  "gpu_total_gib": 14.56317138671875,
  "dtype": "torch.bfloat16",
  "output_shape_per_sequence": [
    512,
    16
  ],
  "ground_truth_loaded": false,
  "updated_utc": "2026-08-18T18:16:54.390404+00:00",
  "artifact

,sequence_id,split,recording_name,in_current_cohort,complete,embedding_path,metadata_path,generation_seconds,attempt,media_mode,min_column_norm,max_column_norm
0,disassembly_nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,validation,nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,True,True,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/pseudotext_embeddings/disassembly_nusar-2021_action_both_90...,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/parsed_predictions/disassembly_nusar-2021_action_both_9034-...,76.754146,1,ordered_images,1.0,1.0
1,assembly_nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,validation,nusar-2021_action_both_9034-c02b_9034_user_id_2021-02-23_173828,True,True,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/pseudotext_embeddings/assembly_nusar-2021_action_both_9034-...,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/parsed_predictions/assembly_nusar-2021_action_both_9034-c02...,67.130657,1,ordered_images,1.0,1.0
2,disassembly_nusar-2021_action_both_9046-b06b_9046_user_id_2021-02-22_105953,validation,nusar-2021_action_both_9046-b06b_9046_user_id_2021-02-22_105953,True,True,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/pseudotext_embeddings/disassembly_nusar-2021_action_both_90...,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/parsed_predictions/disassembly_nusar-2021_action_both_9046-...,66.663461,1,ordered_images,1.0,1.0
3,assembly_nusar-2021_action_both_9046-b06b_9046_user_id_2021-02-22_105953,validation,nusar-2021_action_both_9046-b06b_9046_user_id_2021-02-22_105953,True,True,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/pseudotext_embeddings/assembly_nusar-2021_action_both_9046-...,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/parsed_predictions/assembly_nusar-2021_action_both_9046-b06...,67.905743,1,ordered_images,1.0,1.0
4,disassembly_nusar-2021_action_both_9022-a18_9022_user_id_2021-02-23_104757,validation,nusar-2021_action_both_9022-a18_9022_user_id_2021-02-23_104757,True,True,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/pseudotext_embeddings/disassembly_nusar-2021_action_both_90...,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/parsed_predictions/disassembly_nusar-2021_action_both_9022-...,69.446201,1,ordered_images,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
115,assembly_nusar-2021_action_both_9043-b05a_9043_user_id_2021-02-05_134455,validation,nusar-2021_action_both_9043-b05a_9043_user_id_2021-02-05_134455,True,True,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/pseudotext_embeddings/assembly_nusar-2021_action_both_9043-...,/con

,model_class_id,action_cls,predicted_steps
0,0.0,inspect toy,1336
1,1.0,attach cabin,178
4,4.0,attach wheel,162
5,5.0,screw chassis,54
8,8.0,attach interior,25
16,16.0,attach base,24
27,78.0,screw body,16
3,3.0,detach wheel,15
19,23.0,inspect chassis,15
13,13.0,attach body,9


## 15. Release the VLM and report the Notebook 25 hand-off


In [53]:
del model
del processor
gc.collect()
torch.cuda.empty_cache()

print("GPU allocated after release:", f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
print("Pseudo-text embeddings:", EMBEDDING_DIR)
print("Sequence status:", MANIFEST_DIR / "sequence_status.csv")
print("Step predictions:", MANIFEST_DIR / "step_predictions.csv")
print("Summary:", OUT_ROOT / "final_summary.json")


GPU allocated after release: 0.01 GiB
Pseudo-text embeddings: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/pseudotext_embeddings
Sequence status: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/manifests/sequence_status.csv
Step predictions: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/manifests/step_predictions.csv
Summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/24_qwen35_pseudotext/qwen35_2b_closedset_temporal_v1/final_summary.json
